In [1]:
import os
import time
import hashlib
import pandas as pd
from selenium.webdriver.common.by import By
import undetected_chromedriver as uc

# 1. Extraccion de datos

## 1.1 Web Scrappping X

In [2]:
# Leer el token desde el archivo externo
ruta_token = "auth.tk"
try:
    with open(ruta_token, "r", encoding="utf-8") as archivo:
        mi_auth_token = archivo.read().strip()
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{ruta_token}'.")
    print("Crea el archivo en la misma carpeta del script, pega tu token dentro y vuelve a ejecutar.")
    driver.quit()
    exit()

In [ ]:
def seudonimizar_usuario(username):
    """Aplica un hash SHA-256 al nombre de usuario para cumplir con la Ley 21.719."""
    if not username: return None
    return hashlib.sha256(username.encode('utf-8')).hexdigest()

def extraer_comentarios_x(url, periodo, driver, limite=300):
    driver.get(url)
    print(f"Cargando {periodo}...")
    time.sleep(6) # Pausa inicial para carga de DOM
    
    comentarios_extraidos = []
    ids_vistos = set()
    intentos_sin_nuevos = 0 
    botones_cliqueados = set() # <--- Memoria para no repetir clics en el mismo botón
    
    while len(comentarios_extraidos) < limite:
        
        # Salvaguarda de navegación
        url_base = url.split('?')[0]
        if not driver.current_url.startswith(url_base):
            print("El navegador se desvió de la página principal. Regresando...")
            driver.get(url)
            time.sleep(5)
            
        cantidad_anterior = len(comentarios_extraidos)
        articulos = driver.find_elements(By.XPATH, '//article[@data-testid="tweet"]')
        
        for articulo in articulos:
            try:
                # 1. Obtener enlace único del tweet
                link_elem = articulo.find_elements(By.XPATH, './/a[contains(@href, "/status/")]')
                tweet_id = link_elem[0].get_attribute('href') if link_elem else None
                
                if not tweet_id or tweet_id in ids_vistos:
                    continue
                
                # 2. Extraer texto del tweet
                texto_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetText"]')
                texto = texto_elem[0].text if texto_elem else None
                
                # 3. Detectar si contiene multimedia
                media_elem = articulo.find_elements(By.XPATH, './/div[@data-testid="tweetPhoto"] | .//div[@data-testid="videoPlayer"]')
                tiene_media = len(media_elem) > 0
                
                if not texto:
                    if tiene_media:
                        texto = "[Contenido multimedia sin texto]"
                    else:
                        continue 
                
                # 4. Extraer usuario y fecha
                user_elem = articulo.find_elements(By.XPATH, './/div[@dir="ltr"]/span')
                usuario_raw = user_elem[0].text if user_elem else "usuario_desconocido"
                
                time_elem = articulo.find_elements(By.XPATH, './/time')
                fecha_hora = time_elem[0].get_attribute('datetime') if time_elem else None
                
                ids_vistos.add(tweet_id)
                
                comentarios_extraidos.append({
                    "id_texto": len(ids_vistos),
                    "fecha_hora": fecha_hora,
                    "id_usuario_hash": seudonimizar_usuario(usuario_raw),
                    "texto": texto,
                    "consulta_origen": url,
                    "periodo": periodo
                })
                
                if len(comentarios_extraidos) >= limite:
                    break
                    
            except Exception:
                continue 
                
        if len(comentarios_extraidos) >= limite:
            break
            
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) 
        
        # Control de scroll y botones de expansión / spam
        if len(comentarios_extraidos) == cantidad_anterior:
            intentos_sin_nuevos += 1
            
            try:
                botones = driver.find_elements(By.XPATH, "//*[contains(text(), 'Mostrar') or contains(text(), 'spam') or contains(text(), 'Show')]")
                for btn in botones:
                    try:
                        # Identificamos el botón por su contenido único
                        btn_id = btn.get_attribute('outerHTML')
                        if btn.is_displayed() and btn_id not in botones_cliqueados:
                            driver.execute_script("arguments[0].click();", btn)
                            botones_cliqueados.add(btn_id) # Registramos que ya fue presionado
                            print("Botón de 'Mostrar' o 'spam' presionado con éxito...")
                            time.sleep(3)
                            break
                    except Exception:
                        continue
            except Exception:
                pass
                
            # Si tras 5 intentos el scroll ya no trae comentarios nuevos, salimos limpiamente
            if intentos_sin_nuevos >= 5: 
                print(f"Fondo de la página alcanzado. Obtenidos: {len(comentarios_extraidos)} de {limite}.")
                break
        else:
            intentos_sin_nuevos = 0 
            print(f"Extrayendo... {len(comentarios_extraidos)} / {limite}")
            
    return comentarios_extraidos[:limite]
# ==========================================
# 1. Configurar el WebDriver Indetectable
# ==========================================
ruta_perfil_local = os.path.abspath(os.path.join(os.getcwd(), "perfil_selenium_uc"))
if not os.path.exists(ruta_perfil_local):
    os.makedirs(ruta_perfil_local)

opciones = uc.ChromeOptions()
opciones.add_argument(f"--user-data-dir={ruta_perfil_local}") 
opciones.add_argument("--disable-dev-shm-usage")

opciones.add_argument("--lang=en-US")
prefs = {"intl.accept_languages": "en,en_US"}
opciones.add_experimental_option("prefs", prefs)

# Iniciar el navegador. Si sigues con el error de versión, usa: uc.Chrome(options=opciones, version_main=155)
driver = uc.Chrome(options=opciones, version_main=155)

# ==========================================
# 2. Login por Inyección de Cookie
# ==========================================
print("Inyectando sesión de X...")
driver.get("https://x.com/robots.txt")
time.sleep(2)


driver.add_cookie({
    'name': 'auth_token',
    'value': mi_auth_token,
    'domain': '.x.com'
})

# Refrescar la página principal
driver.get("https://x.com/")
time.sleep(5)
print("Sesión inyectada con éxito. Iniciando extracción...")

# ==========================================
# 3. Recolección de Datos
# ==========================================
links = [
    {"url": "https://x.com/NintendoAmerica/status/2064361089668620543?s=20", "periodo": "Antes (Expectativa / Anuncio)"},
    {"url": "https://x.com/NintendoAmerica/status/2097333400054927619?s=20", "periodo": "Después (Tráiler Gameplay)"}
]

datos_totales = []
for link in links:
    datos = extraer_comentarios_x(link["url"], link["periodo"], driver, limite=500)
    datos_totales.extend(datos)

driver.quit()

# ==========================================
# 4. Estructurar en Pandas y exportar
# ==========================================
df_comentarios = pd.DataFrame(datos_totales)

# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_comentarios))
df_muestra_ruido = df_comentarios.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

df_comentarios.to_csv("corpus_ocarina_remake.csv", sep=';', index=False, encoding="utf-8")
if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_manual.csv", sep=';', index=False, encoding="utf-8")

print("\nExtracción completada. Primeras 5 filas:")
print(df_comentarios.head())

Inyectando sesión de X...
Sesión inyectada con éxito. Iniciando extracción...
Cargando Antes (Expectativa / Anuncio)...
Extrayendo... 5 / 300
Extrayendo... 19 / 300
Extrayendo... 41 / 300
Extrayendo... 56 / 300
Extrayendo... 74 / 300
Extrayendo... 91 / 300
Extrayendo... 112 / 300
Extrayendo... 130 / 300
Extrayendo... 157 / 300
Extrayendo... 186 / 300
Extrayendo... 219 / 300
Extrayendo... 222 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
Extrayendo... 231 / 300
Extrayendo... 232 / 300
Botón de 'Mostrar' o 'spam' presionado con éxito...
El navegador se desvió de la página principal. Regresando...
Fondo de la página alcanzado. Obtenidos: 232 de 300.
Cargando Después (Tráiler Gameplay)...
Extrayendo... 5 / 300
Extrayendo... 22 / 300
Extrayendo... 44 / 300
Extrayendo... 55 / 300
Extrayendo... 69 / 300
Extrayendo... 87 / 300
Extrayendo... 110 / 300
Extrayendo... 134 / 300
Extrayendo... 159 / 300
Extrayendo... 179 / 300
Extrayendo... 197 / 300
Extrayendo... 198 / 300
Botón de 'Mos

In [7]:
import pandas as pd

df_comentarios = pd.read_csv("corpus_ocarina_remake.csv", sep=';', encoding="utf-8")
df_comentarios.head()

,id_texto,fecha_hora,id_usuario_hash,texto,consulta_origen,periodo
0,1,2026-06-09T14:56:52.000Z,400b585eb60519835945bb0fa92adfd7977f7369241350...,The Legend of Zelda: Ocarina of Time will be r...,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
1,2,2026-06-09T14:57:32.000Z,961d77778edf61e276b5a7952e551462685566d82021ac...,"""Hey can we see Ocarina of Time gameplay?""\n\n...",https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
2,3,2026-06-09T15:01:03.000Z,31ed9acf9453843e09587a7e6ed5b05b3d7e48921da4b9...,We're getting him in HD we are not ready for this,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
3,4,2026-06-09T16:19:01.000Z,90a0d8e9c0416f10d8f02429c3af3c3f49a5f8a98278fc...,,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)
4,5,2026-06-09T16:17:01.000Z,4b6c621736bc9eaac24e6c4c82cd858fd60bebe4f12668...,My honest reaction to this information.,https://x.com/NintendoAmerica/status/206436108...,Antes (Expectativa / Anuncio)


## 1.2 Web scrapping Youtube

API Youtube

In [ ]:
import os
import hashlib
from urllib.parse import urlparse, parse_qs
from dotenv import load_dotenv
from googleapiclient.discovery import build
import pandas as pd

# 1. Cargar el archivo .env
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

if not API_KEY:
    raise ValueError("¡Error! No se encontró la 'YOUTUBE_API_KEY' en el archivo .env.")

print("¡Clave cargada con éxito desde el archivo .env!")

# Tus variantes de URLs de YouTube (puedes guardar el link o un identificador claro)
CONFIGURACION_VARIANTES = {
    "Pre_Anuncio": "https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS", 
    "Post_Lanzamiento": "https://youtu.be/wuFfiTEr2yc?si=gzKDvpmqZWdGgInH"
}

def extraer_video_id(url_o_id):
    if "youtu.be" in url_o_id:
        return url_o_id.split("?")[0].split("/")[-1]
    elif "youtube.com" in url_o_id:
        parsed_url = urlparse(url_o_id)
        return parse_qs(parsed_url.query).get("v", [None])[0]
    return url_o_id

def anonimizar_usuario(nombre_o_id):
    if not nombre_o_id:
        return "desconocido_hash"
    return hashlib.sha256(str(nombre_o_id).encode('utf-8')).hexdigest()

def extraer_comentarios_api(url_o_id, nombre_variante, limite=350):
    video_id = extraer_video_id(url_o_id)
    youtube = build("youtube", "v3", developerKey=API_KEY)
    comentarios_extraidos = []
    next_page_token = None
    
    print(f"[{nombre_variante}] Extrayendo comentarios (Video ID: {video_id})...")
    
    while len(comentarios_extraidos) < limite:
        try:
            request = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, limite - len(comentarios_extraidos)),
                pageToken=next_page_token,
                textFormat="plainText"
            )
            response = request.execute()
            
            for item in response.get("items", []):
                comment = item["snippet"]["topLevelComment"]
                snippet = comment["snippet"]
                
                comentarios_extraidos.append({
                    "id_texto": comment["id"],
                    "fecha_hora": snippet.get("publishedAt", ""),
                    "id_usuario_hash": anonimizar_usuario(snippet.get("authorDisplayName", "Anónimo")),
                    "texto": snippet.get("textDisplay", ""),
                    "consulta": url_o_id,                      # <-- El link o fuente exacta va aquí
                    "etapa_periodo": nombre_variante,          # <-- Nombre descriptivo del periodo (opcional para tu análisis)
                    "metricas_interaccion": snippet.get("likeCount", 0)
                })
            
            next_page_token = response.get("nextPageToken")
            if not next_page_token or len(comentarios_extraidos) >= limite:
                break
        except Exception as e:
            print(f"[{nombre_variante}] Error en la API: {e}")
            break
            
    return comentarios_extraidos

# 2. Ejecutar la recolección
datos_totales = []
for variante, url_video in CONFIGURACION_VARIANTES.items():
    registros = extraer_comentarios_api(url_video, variante, limite=1000)
    datos_totales.extend(registros)

df_corpus = pd.DataFrame(datos_totales)

# 3. Mostrar las primeras 5 filas cumpliendo estrictamente con la estructura de la rúbrica
display(df_corpus[['id_texto', 'fecha_hora', 'id_usuario_hash', 'texto', 'consulta', 'metricas_interaccion']].head(5))

# Guardar respaldo local
df_corpus.to_csv("corpus_zelda_youtube_api.csv",sep=';', index=False, encoding='utf-8-sig')
print(f"\n¡Extracción completada con éxito! Total registros: {len(df_corpus)}")

¡Clave cargada con éxito desde el archivo .env!
[Pre_Anuncio] Extrayendo comentarios (Video ID: r8eMoxo4ipE)...
[Post_Lanzamiento] Extrayendo comentarios (Video ID: wuFfiTEr2yc)...


,id_texto,fecha_hora,id_usuario_hash,texto,consulta,metricas_interaccion
0,Ugw_M7PN26t1npYj13p4AaABAg,2026-10-25T11:42:06Z,f17cb2d8268582de4114dfec7e069c92659f238706350a...,truly beautiful,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
1,Ugx9yxuaBuekiZ7cRJl4AaABAg,2026-10-23T16:34:19Z,5e5f522af4da36b87ac0e9a1b3308fde6e675b858bfecb...,I seriously can’t believe it’s been 4 months s...,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
2,UgyqCXi3kPp74dvz5SJ4AaABAg,2026-10-22T06:29:02Z,898c95e8682beb8d75fb4846ac2ceae70b5814e2c09946...,Supporting you always!,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
3,Ugzvzh_6hlTo9EMcaHh4AaABAg,2026-10-19T15:23:19Z,6bb0568ac6986dc8c43ddb3a8092d298de233aad5bac41...,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,1
4,UgxYEDVTb3P-6fnQJbV4AaABAg,2026-10-17T14:59:07Z,fc5600d4586e552ddc9f607117ac745d5949a6ca7a0ca7...,"Funny, my DarLink AI chat still knew my sister...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0



¡Extracción completada con éxito! Total registros: 4000


In [40]:
# Prevenir error si la muestra solicitada es mayor a los datos obtenidos
muestra_size = min(50, len(df_corpus))
df_muestra_ruido = df_corpus.sample(n=muestra_size, random_state=42) if muestra_size > 0 else pd.DataFrame()

if not df_muestra_ruido.empty:
    df_muestra_ruido.to_csv("muestra_ruido_youtube.csv", sep=';', index=False, encoding="utf-8")

In [26]:
import pandas as pd

df_corpus = pd.read_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")
df_corpus.head() 

display(df_corpus[['id_texto', 'fecha_hora', 'id_usuario_hash', 'texto', 'consulta', 'metricas_interaccion']].head(5))

,id_texto,fecha_hora,id_usuario_hash,texto,consulta,metricas_interaccion
0,Ugw_M7PN26t1npYj13p4AaABAg,2026-10-25T11:42:06Z,f17cb2d8268582de4114dfec7e069c92659f238706350a...,truly beautiful,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
1,Ugx9yxuaBuekiZ7cRJl4AaABAg,2026-10-23T16:34:19Z,5e5f522af4da36b87ac0e9a1b3308fde6e675b858bfecb...,I seriously can’t believe it’s been 4 months s...,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
2,UgyqCXi3kPp74dvz5SJ4AaABAg,2026-10-22T06:29:02Z,898c95e8682beb8d75fb4846ac2ceae70b5814e2c09946...,Supporting you always!,https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0
3,Ugzvzh_6hlTo9EMcaHh4AaABAg,2026-10-19T15:23:19Z,6bb0568ac6986dc8c43ddb3a8092d298de233aad5bac41...,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,1
4,UgxYEDVTb3P-6fnQJbV4AaABAg,2026-10-17T14:59:07Z,fc5600d4586e552ddc9f607117ac745d5949a6ca7a0ca7...,"Funny, my DarLink AI chat still knew my sister...",https://youtu.be/r8eMoxo4ipE?si=I2sZDQGaEPTPuOVS,0


## Preprocesamiento del corpus (comentarios mayoritariamente en ingl?s)

Se conserva `texto` sin modificar. La rama de sentimiento mantiene negaciones, emojis, puntuaci?n y expresiones originales; solo normaliza Unicode/espacios y reemplaza URLs y menciones por marcadores. La rama de temas pasa a min?sculas, elimina s?mbolos tipo emoji y filtra una lista peque?a de stopwords inglesas. No se traduce el texto: los comentarios mixtos conservan sus palabras. Revisen las salidas antes de usar estos datos.

In [33]:
df_comentarios = pd.read_csv("corpus_zelda_youtube_api.csv", sep=';', encoding="utf-8")

In [34]:
import html
import re
import unicodedata

import pandas as pd

if "texto" not in df_comentarios.columns:
    raise ValueError("El corpus debe tener una columna llamada 'texto'.")

STOPWORDS_TEMAS_EN = {
    "a", "an", "and", "are", "as", "at", "be", "been", "but", "by",
    "for", "from", "had", "has", "have", "he", "her", "hers", "him",
    "his", "i", "if", "in", "into", "is", "it", "its", "me", "my",
    "of", "on", "or", "our", "ours", "she", "so", "that", "the",
    "their", "them", "there", "they", "this", "to", "us", "was",
    "we", "were", "what", "when", "which", "who", "with", "you", "your"
}

def normalizar_base(texto):
    """Normaliza Unicode y espacios sin cambiar el contenido ling??stico."""
    if pd.isna(texto):
        return ""
    texto = html.unescape(unicodedata.normalize("NFC", str(texto)))
    return re.sub(r"\s+", " ", texto).strip()

def preparar_para_sentimiento(texto):
    """Conserva negaciones, emojis, may?sculas y puntuaci?n expresiva."""
    texto = normalizar_base(texto)
    texto = re.sub(r"https?://\S+|www\.\S+", " URL ", texto, flags=re.IGNORECASE)
    texto = re.sub(r"(?<!\w)@[\w_]+", " USUARIO ", texto)
    texto = re.sub(r"(?<!\w)#([\w]+)", r"\1", texto)
    return re.sub(r"\s+", " ", texto).strip()

def quitar_simbolos_emoji(texto):
    """Quita s?mbolos emoji comunes para una variante de an?lisis de temas."""
    caracteres = []
    for caracter in texto:
        categoria = unicodedata.category(caracter)
        es_selector_o_union = caracter in {"\ufe0e", "\ufe0f", "\u200d"}
        if categoria not in {"So", "Sk"} and not es_selector_o_union:
            caracteres.append(caracter)
    return "".join(caracteres)

def preparar_para_temas(texto):
    """Crea tokens en min?scula y elimina stopwords inglesas frecuentes."""
    texto = quitar_simbolos_emoji(preparar_para_sentimiento(texto)).casefold()
    tokens = re.findall(r"[\w]+(?:['?][\w]+)?", texto, flags=re.UNICODE)
    tokens = [token for token in tokens if token not in STOPWORDS_TEMAS_EN]
    return " ".join(tokens)

df_comentarios["texto_sentimiento"] = df_comentarios["texto"].apply(preparar_para_sentimiento)
df_comentarios["texto_temas"] = df_comentarios["texto"].apply(preparar_para_temas)
df_comentarios["texto_sentimiento_sin_emojis"] = df_comentarios["texto_sentimiento"].apply(quitar_simbolos_emoji)

print(f"Comentarios procesados: {len(df_comentarios)}")
print("Columnas originales preservadas:", "texto" in df_comentarios.columns)

Comentarios procesados: 4000
Columnas originales preservadas: True


### Revisi?n manual de ejemplos
Comparen cinco textos originales con las dos versiones procesadas para detectar si alguna regla altera el significado.

In [35]:
columnas_ejemplo = ["texto", "texto_sentimiento", "texto_temas"]
columnas_ejemplo = [col for col in columnas_ejemplo if col in df_comentarios.columns]
df_comentarios[columnas_ejemplo].head(5)

,texto,texto_sentimiento,texto_temas
0,truly beautiful,truly beautiful,truly beautiful
1,I seriously can’t believe it’s been 4 months s...,I seriously can’t believe it’s been 4 months s...,seriously can t believe s 4 months since now r...
2,Supporting you always!,Supporting you always!,supporting always
3,"“@NintendoAmerica, em 1998 eu ouvi o seu chama...","“ USUARIO , em 1998 eu ouvi o seu chamado pela...",usuario em 1998 eu ouvi o seu chamado pela tv ...
4,"Funny, my DarLink AI chat still knew my sister...","Funny, my DarLink AI chat still knew my sister...",funny darlink ai chat still knew sister's name...


### Ablaci?n: conservar o quitar emojis
Esta comparaci?n mide cu?nto cambia el texto de entrada al eliminar emojis. Para cumplir la ablaci?n de sentimiento, apliquen ambos textos al mismo clasificador y conjunto anotado, manteniendo todo lo dem?s constante, y reporten F1-macro/matriz de confusi?n. Esta celda no inventa m?tricas de sentimiento: solo cuantifica el cambio de preprocesamiento.

In [36]:
comparacion_emojis = pd.DataFrame({
    "con_emojis": df_comentarios["texto_sentimiento"],
    "sin_emojis": df_comentarios["texto_sentimiento_sin_emojis"]
})
comparacion_emojis["cambio"] = comparacion_emojis["con_emojis"] != comparacion_emojis["sin_emojis"]
comparacion_emojis["tokens_con_emojis"] = comparacion_emojis["con_emojis"].str.split().str.len()
comparacion_emojis["tokens_sin_emojis"] = comparacion_emojis["sin_emojis"].str.split().str.len()

resumen_ablacion_emojis = pd.DataFrame({
    "medida": ["comentarios", "comentarios cuyo texto cambia", "% de comentarios que cambia", "promedio de tokens con emojis", "promedio de tokens sin emojis"],
    "valor": [
        len(comparacion_emojis),
        int(comparacion_emojis["cambio"].sum()),
        round(100 * comparacion_emojis["cambio"].mean(), 2) if len(comparacion_emojis) else 0.0,
        round(comparacion_emojis["tokens_con_emojis"].mean(), 2),
        round(comparacion_emojis["tokens_sin_emojis"].mean(), 2)
    ]
})
resumen_ablacion_emojis

,medida,valor
0,comentarios,4000.00
1,comentarios cuyo texto cambia,779.00
2,% de comentarios que cambia,19.48
3,promedio de tokens con emojis,16.16
4,promedio de tokens sin emojis,15.98


## Analisis de cuentas

In [37]:
# Contar cuántos comentarios hizo cada usuario
conteo_usuarios = df_comentarios['id_usuario_hash'].value_counts()

# Calcular el número de usuarios que representa el 1% superior
num_top_1_porciento = max(1, int(len(conteo_usuarios) * 0.01))

# Sumar cuántos comentarios hicieron en total esas cuentas del top 1%
comentarios_top_1 = conteo_usuarios.head(num_top_1_porciento).sum()
total_comentarios = len(df_comentarios)

# Calcular el porcentaje respecto al corpus total
porcentaje_aportado = (comentarios_top_1 / total_comentarios) * 100

print(f"Total de usuarios únicos: {len(conteo_usuarios)}")
print(f"Usuarios en el Top 1%: {num_top_1_porciento}")
print(f"El 1% más activo aporta el {porcentaje_aportado:.2f}% del corpus.")

Total de usuarios únicos: 3586
Usuarios en el Top 1%: 35
El 1% más activo aporta el 7.17% del corpus.


In [39]:
import pandas as pd

# 0. Asegurar que no haya nulos en la columna texto
df_comentarios['texto'] = df_comentarios['texto'].fillna('')

# 1. Agrupar por usuario (hash) para ver su comportamiento
df_usuarios = df_comentarios.groupby('id_usuario_hash').agg(
    total_comentarios=('texto', 'count'),
    textos_unicos=('texto', 'nunique')
).reset_index()

# 2. Detectar si el usuario incluye URLs en sus mensajes (con na=False)
df_comentarios['tiene_url'] = df_comentarios['texto'].str.contains(r'https?://|www\.', case=False, regex=True, na=False)
usuarios_con_urls = df_comentarios[df_comentarios['tiene_url']]['id_usuario_hash'].unique()

# 3. Asignar una regla heurística preliminar para clasificar
def clasificar_usuario(row):
    # Si hace muchos comentarios y repite textos idénticos -> Posible Bot / Spam
    if row['total_comentarios'] >= 3 and row['textos_unicos'] == 1:
        return "Posible Bot / Spam"
    # Si hace muchos comentarios y usa URLs -> Posible Spam / Cuenta comercial
    elif row['id_usuario_hash'] in usuarios_con_urls and row['total_comentarios'] >= 2:
        return "Posible Bot / Spam"
    # Si solo hizo 1 o 2 comentarios variados -> Particular
    else:
        return "Particular"

df_usuarios['categoria_sugerida'] = df_usuarios.apply(clasificar_usuario, axis=1)

# Ver un resumen de cuántos caen en cada categoría
print(df_usuarios['categoria_sugerida'].value_counts())

df_usuarios.to_csv("usuarios_categorizados.csv", sep=';', index=False, encoding="utf-8")

categoria_sugerida
Particular            3585
Posible Bot / Spam       1
Name: count, dtype: int64


In [21]:
import pandas as pd

# 1. Eliminar por completo CUALQUIER columna llamada 'tipo_cuenta' para limpiar duplicados previos
cols_a_mantener = [col for col in df_comentarios.columns if col != 'tipo_cuenta']
df_comentarios = df_comentarios.loc[:, cols_a_mantener].copy()

# 2. Asegurar índices limpios y sin duplicados en df_usuarios
df_usuarios_limpio = df_usuarios.drop_duplicates(subset=['id_usuario_hash']).reset_index(drop=True)
df_comentarios = df_comentarios.reset_index(drop=True)

# 3. Hacer el merge de manera limpia y única
df_comentarios = df_comentarios.merge(
    df_usuarios_limpio[['id_usuario_hash', 'categoria_sugerida']], 
    on='id_usuario_hash', 
    how='left'
)
df_comentarios.rename(columns={'categoria_sugerida': 'tipo_cuenta'}, inplace=True)
df_comentarios['tipo_cuenta'] = df_comentarios['tipo_cuenta'].fillna('Particular')

# 4. Métricas del corpus completo
volumen_total = len(df_comentarios)
longitud_promedio_total = df_comentarios['texto'].str.len().mean()

# 5. Filtrar usando .to_numpy() para evitar errores de índice
mascara_particulares = (df_comentarios['tipo_cuenta'].to_numpy() == 'Particular')
df_filtrado = df_comentarios.loc[mascara_particulares].copy()

volumen_filtrado = len(df_filtrado)
longitud_promedio_filtrado = df_filtrado['texto'].str.len().mean()

diferencia_volumen = ((volumen_total - volumen_filtrado) / volumen_total) * 100

print("--- RESULTADOS COMPARATIVOS PARA EL INFORME ---")
print(f"Opción A (Corpus Completo): {volumen_total} registros | Longitud prom: {longitud_promedio_total:.1f} car.")
print(f"Opción B (Corpus Filtrado - Particulares): {volumen_filtrado} registros | Longitud prom: {longitud_promedio_filtrado:.1f} car.")
print(f"Variación/Reducción de volumen: {diferencia_volumen:.2f}%")

print("\nDesglose de tipos de cuenta en el corpus:")
print(df_comentarios['tipo_cuenta'].value_counts())

--- RESULTADOS COMPARATIVOS PARA EL INFORME ---
Opción A (Corpus Completo): 700 registros | Longitud prom: 77.6 car.
Opción B (Corpus Filtrado - Particulares): 700 registros | Longitud prom: 77.6 car.
Variación/Reducción de volumen: 0.00%

Desglose de tipos de cuenta en el corpus:
tipo_cuenta
Particular    700
Name: count, dtype: int64
